# 1. Gold Layer: Analytics & Anomaly Detection

This notebook builds analytics-ready tables from the Silver layer.

Key outputs:
- Anomaly detection (time-series logic)
- Machine performance metrics

The Gold layer provides business insights and monitoring capabilities.

## 2. Load Cleaned Data

Reading structured and validated data from the Silver layer.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, max, col, when

df = spark.table("main.default.silver_machine_readings")

display(df.limit(5))

machine_id,pressure,temperature,timestamp,vibration
M5,3.69,65.0,2026-04-12T10:52:22.060Z,0.39
M2,1.48,61.0,2026-04-12T10:52:22.060Z,0.22
M3,3.07,66.0,2026-04-12T10:52:22.061Z,0.9
M1,2.18,85.0,2026-04-12T10:52:22.061Z,0.85
M2,4.58,74.0,2026-04-12T10:52:22.062Z,0.14


## 3. Rolling Average Calculation

Compute rolling average temperature per machine using a window function.

- Partition: machine_id
- Order: timestamp
- Window: last 5 records

In [0]:
windowSpec = Window.partitionBy("machine_id") \
    .orderBy("timestamp") \
    .rowsBetween(-5, 0)

df_with_avg = df.withColumn(
    "rolling_avg_temp",
    avg("temperature").over(windowSpec)
)

## 4. Anomaly Detection

Rules:
- HIGH_TEMP → temperature ≥ 88
- SPIKE → temperature > rolling_avg × 1.15

In [0]:
df_anomaly = df_with_avg.withColumn(
    "is_anomaly",
    when(col("temperature") >= 88, True)
    .when(col("temperature") > col("rolling_avg_temp") * 1.15, True)
    .otherwise(False)
)

df_anomaly = df_anomaly.withColumn(
    "anomaly_type",
    when(col("temperature") >= 88, "HIGH_TEMP")
    .when(col("temperature") > col("rolling_avg_temp") * 1.15, "SPIKE")
    .otherwise("NORMAL")
)

## 5. Save Gold Table: Anomalies

Stores detected anomalies for monitoring and alerting.

In [0]:
df_anomaly.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("main.default.gold_machine_anomalies")

## 6. Machine Performance Metrics

Aggregated metrics per machine:
- Average temperature
- Maximum temperature
- Average vibration

In [0]:
df_metrics = df.groupBy("machine_id").agg(
    avg("temperature").alias("avg_temp"),
    max("temperature").alias("max_temp"),
    avg("vibration").alias("avg_vibration")
)

## 7. Save Gold Table: Machine Metrics

In [0]:
df_metrics.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("main.default.gold_machine_metrics")

## 8. Verification: Detected Anomalies

In [0]:
spark.sql("""
SELECT *
FROM main.default.gold_machine_anomalies
WHERE is_anomaly = true
LIMIT 10
""").display()

machine_id,pressure,temperature,timestamp,vibration,rolling_avg_temp,is_anomaly,anomaly_type
M1,1.46,89.0,2026-04-12T10:52:22.064Z,0.2,78.66666666666667,true,HIGH_TEMP
M1,1.36,90.0,2026-04-12T10:52:22.066Z,0.17,80.33333333333333,true,HIGH_TEMP
M1,2.51,85.0,2026-04-12T10:52:22.067Z,0.44,70.0,true,SPIKE
M1,3.35,87.0,2026-04-12T10:52:22.067Z,0.17,74.16666666666667,true,SPIKE
M2,4.81,89.0,2026-04-12T10:52:22.060Z,0.49,79.0,true,HIGH_TEMP
M2,4.99,88.0,2026-04-12T10:52:22.062Z,0.1,74.33333333333333,true,HIGH_TEMP
M2,4.44,88.0,2026-04-12T10:52:22.064Z,0.85,75.66666666666667,true,HIGH_TEMP
M2,1.49,86.0,2026-04-12T10:52:22.066Z,0.61,69.66666666666667,true,SPIKE
M2,2.12,87.0,2026-04-12T10:52:22.066Z,0.99,73.5,true,SPIKE
M2,3.45,84.0,2026-04-12T10:52:22.067Z,0.39,71.83333333333333,true,SPIKE


## 9. SQL Analysis: Machine Temperature Overview

In [0]:
%sql
SELECT 
  machine_id, 
  MAX(temperature) as max_actual_temp,
  AVG(temperature) as avg_actual_temp
FROM main.default.silver_machine_readings
GROUP BY machine_id

machine_id,max_actual_temp,avg_actual_temp
M5,90.0,74.72839506172839
M2,89.0,75.72619047619048
M3,89.0,75.63333333333334
M1,90.0,76.80281690140845
M4,90.0,76.20270270270271


## 10. SQL Analysis: Anomaly Insights

Shows:
- number of incidents
- average temperature during anomalies

In [0]:
%sql
SELECT 
  machine_id,
  anomaly_type,
  COUNT(*) as incident_count,
  ROUND(AVG(temperature), 2) as avg_temp_during_incident
FROM main.default.gold_machine_anomalies
WHERE is_anomaly = true
GROUP BY machine_id, anomaly_type
ORDER BY incident_count DESC

machine_id,anomaly_type,incident_count,avg_temp_during_incident
M4,HIGH_TEMP,6,89.67
M5,SPIKE,6,86.0
M3,HIGH_TEMP,5,88.4
M2,HIGH_TEMP,5,88.6
M5,HIGH_TEMP,3,89.33
M2,SPIKE,3,85.67
M4,SPIKE,3,84.33
M1,HIGH_TEMP,2,89.5
M1,SPIKE,2,86.0
M3,SPIKE,2,85.0


In [0]:
print("Anomalies count:", df_anomaly.filter("is_anomaly = true").count())

Anomalies count: 37
